# I. FULL DATASET / 5 clusters (mcs 1900)

## 1. Decision tree for internal variables

In [1]:
# ── 0. Config ──────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree, _tree
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

plt.style.use('default')  # white background for all figures

# ── Paths ──────────────────────────────────────────────────────────────────────

CSV_PATH = "Results/Regular_clustering/Full_dataset/s2_noweights/clustering_mcs1900.csv"
OUT_DIR  = "Results/Regular_clustering/Full_dataset/s2_noweights/description_mcs1900/decision_tree"

os.makedirs(OUT_DIR, exist_ok=True)

# ── Load dataset ───────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH, low_memory=False)

print(f"Dataset loaded : {len(df)} rows, {df.shape[1]} columns")

# ── 1. Feature definition ──────────────────────────────────────────────────────

IMAGING_COLS_BOOL = {
    "has_ultrasound", "has_ct_scan", "has_xray",
    "has_mri", "has_radio_interventional", "has_nuclear_medicine",
}

BIO_EXAMS = {
    "has_blood_test", "has_culture",
    "has_lumbar_puncture", "has_blood_gas",
}

PROCEDURE_COLS   = {"had_ekg"}

DISPOSITION_COLS = {
    "hospitalization", "observation_unit", "inter_facility_transfer",
}

COLS_QUANTI  = ["imaging_exam_count", "bio_exam_count"]
COLS_BOOL    = list(IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS)
ALL_FEATURES = COLS_BOOL + COLS_QUANTI

# ── 2. Prepare datasets ────────────────────────────────────────────────────────

df_model = df[ALL_FEATURES + ['cluster']].dropna()

# Tree 1 : clusters only (outliers removed)
df_clusters = df_model[df_model['cluster'] != -1]
X_clusters  = df_clusters[ALL_FEATURES]
y_clusters  = df_clusters['cluster']

# Tree 2 : outlier detection (everyone, binary target)
df_outliers               = df_model.copy()
df_outliers['is_outlier'] = (df_outliers['cluster'] == -1).astype(int)
X_outliers                = df_outliers[ALL_FEATURES]
y_outliers                = df_outliers['is_outlier']

print(f"\n── Tree 1 : cluster structure ──")
print(f"Individuals : {len(df_clusters)}")
print(f"Cluster distribution :\n{y_clusters.value_counts().sort_index()}")

print(f"\n── Tree 2 : outlier detection ──")
print(f"Individuals : {len(df_outliers)}")
print(f"Outliers : {y_outliers.sum()} ({y_outliers.mean():.1%} of total)")

# ── 3. Helper : export tree image ─────────────────────────────────────────────

def export_tree_image(tree, feature_names, class_names, title, filename):
    fig, ax = plt.subplots(figsize=(32, 14), facecolor='white')
    ax.set_facecolor('white')
    plot_tree(
        tree,
        feature_names=feature_names,
        class_names=class_names,
        filled=True,
        rounded=True,
        fontsize=8,
        ax=ax
    )
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{filename}.pdf", format="pdf", dpi=150,
                bbox_inches='tight', facecolor='white')
    plt.savefig(f"{OUT_DIR}/{filename}.png", format="png", dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Exported : {filename}.pdf and {filename}.png")

# ── 4. Helper : export feature importance ─────────────────────────────────────

def export_feature_importance(tree, feature_names, title, filename):
    importances = pd.Series(tree.feature_importances_, index=feature_names)
    importances = importances[importances > 0].sort_values(ascending=False)
    print(f"\nFeature importance ({title}) :")
    print(importances.round(3))
    fig, ax = plt.subplots(figsize=(8, 6), facecolor='white')
    ax.set_facecolor('white')
    importances.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel("Importance (Gini)")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{filename}.png", dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Exported : {filename}.png")

# ── 5. Helper : extract rules leading to a target class ───────────────────────

def get_target_rules(tree, feature_names, target_class):
    tree_      = tree.tree_
    classes    = tree.classes_
    feat_names = [
        feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined"
        for i in tree_.feature
    ]
    rules = []

    def recurse(node, conditions):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name      = feat_names[node]
            threshold = tree_.threshold[node]
            recurse(tree_.children_left[node],  conditions + [f"{name} <= {threshold:.2f}"])
            recurse(tree_.children_right[node], conditions + [f"{name} >  {threshold:.2f}"])
        else:
            predicted_class = classes[np.argmax(tree_.value[node])]
            n_samples       = int(tree_.n_node_samples[node])
            purity          = float(np.max(tree_.value[node]) / n_samples)
            if predicted_class == target_class:
                rules.append({
                    'conditions': conditions,
                    'n_samples' : n_samples,
                    'purity'    : purity,
                })

    recurse(0, [])
    return rules

# ══════════════════════════════════════════════════════════════════════════════
# TREE 1 — Cluster structure (outliers excluded)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TREE 1 — Cluster structure")
print("="*60)

tree_clusters = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=300,
    random_state=42
)
tree_clusters.fit(X_clusters, y_clusters)

class_names_clusters = [f"Cluster {c}" for c in sorted(y_clusters.unique())]

export_tree_image(
    tree_clusters,
    ALL_FEATURES,
    class_names_clusters,
    title    = "Decision tree — Cluster structure (Scenario 2)",
    filename = "tree1_clusters"
)

export_feature_importance(
    tree_clusters,
    ALL_FEATURES,
    title    = "Most discriminating features — Cluster structure",
    filename = "tree1_feature_importance"
)

rules_clusters = export_text(tree_clusters, feature_names=ALL_FEATURES)
print("\nText rules :")
print(rules_clusters)

# ══════════════════════════════════════════════════════════════════════════════
# TREE 2 — Outlier detection (full dataset, binary target)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TREE 2 — Outlier detection")
print("="*60)

tree_outliers = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=40,
    random_state=42,
    class_weight='balanced'
)
tree_outliers.fit(X_outliers, y_outliers)

export_tree_image(
    tree_outliers,
    ALL_FEATURES,
    class_names = ["Non-outlier", "Outlier"],
    title       = "Decision tree — Outlier detection (Scenario 2)",
    filename    = "tree2_outliers"
)

export_feature_importance(
    tree_outliers,
    ALL_FEATURES,
    title    = "Most discriminating features — Outlier detection",
    filename = "tree2_feature_importance"
)

rules_outliers = export_text(tree_outliers, feature_names=ALL_FEATURES)
print("\nText rules :")
print(rules_outliers)

# ── Outlier paths ──────────────────────────────────────────────────────────────

outlier_rules = get_target_rules(tree_outliers, ALL_FEATURES, target_class=1)

print("\n── Paths leading to outlier leaves ──")
if not outlier_rules:
    print("No leaf predicts outliers as majority class.")
    print("Try reducing max_depth or min_samples_leaf.")
else:
    for r in outlier_rules:
        print(f"\n  n={r['n_samples']} individuals (purity {r['purity']:.0%})")
        for cond in r['conditions']:
            print(f"    {cond}")

/tmp/ipykernel_434119/3898482661.py:20: DtypeWarning: Columns (75,78,81,85,89,91,93,94,95,97,98,99,100,102,103,104,107,108,109,110,111,113,114,123,125,132,135,136,137) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


Dataset loaded : 29839 rows, 182 columns

── Tree 1 : cluster structure ──
Individuals : 29769
Cluster distribution :
cluster
0    4610
1    9931
2    7416
3    4927
4    2885
Name: count, dtype: int64

── Tree 2 : outlier detection ──
Individuals : 29839
Outliers : 70 (0.2% of total)

TREE 1 — Cluster structure
Exported : tree1_clusters.pdf and tree1_clusters.png

Feature importance (Most discriminating features — Cluster structure) :
bio_exam_count        0.692
imaging_exam_count    0.308
dtype: float64
Exported : tree1_feature_importance.png

Text rules :
|--- imaging_exam_count <= 0.50
|   |--- bio_exam_count <= 0.50
|   |   |--- class: 1
|   |--- bio_exam_count >  0.50
|   |   |--- bio_exam_count <= 1.50
|   |   |   |--- class: 0
|   |   |--- bio_exam_count >  1.50
|   |   |   |--- class: 1
|--- imaging_exam_count >  0.50
|   |--- bio_exam_count <= 0.50
|   |   |--- class: 3
|   |--- bio_exam_count >  0.50
|   |   |--- bio_exam_count <= 1.50
|   |   |   |--- class: 2
|   |   |--- 

In [3]:
# Est-ce que les rares cas de médecine nucléaire sont dans les outliers ?
print(df.groupby('cluster')[['has_nuclear_medicine', 'has_radio_interventional']].mean() * 100)

print(df.groupby('cluster')[['count_ct_scan', 'count_xray', 'count_mri']].mean())

         has_nuclear_medicine  has_radio_interventional
cluster                                                
-1                   0.000000                  4.285714
 0                   0.000000                  0.000000
 1                   0.000000                  0.000000
 2                   0.000000                  0.350593
 3                   0.020296                  0.142074
 4                   0.034662                  0.138648
         count_ct_scan  count_xray  count_mri
cluster                                      
-1            0.857143    0.485714   0.628571
 0            0.000000    0.000000   0.000000
 1            0.000000    0.000000   0.000000
 2            0.570388    0.271036   0.304746
 3            0.272986    0.800690   0.034301
 4            0.689081    0.315078   0.096014
